# Grid Trotter operators

Follow the six register signatures and immediate calls used for real-space grid Trotter
evolution. The signature diagrams describe logical ports. The call graphs describe
the locked resource model; the final example emits a FlagQuantum circuit.

Adapted from [Qualtran v0.7.0](https://github.com/quantumlib/Qualtran/tree/bcb17bb25360ed68ce3bc084645db3e929799389/qualtran/bloqs/chemistry/trotter/grid_ham/trotter.ipynb), Copyright 2023 Google LLC, Apache-2.0.

In [ ]:
from IPython.display import Markdown, SVG, display

from flagquantum.algorithms.primitives.grid_inverse_sqrt import (
    source_grid_inverse_sqrt_newton_cost,
    source_grid_inverse_sqrt_newton_signature,
    source_grid_inverse_sqrt_polynomial_cost,
    source_grid_inverse_sqrt_polynomial_signature,
)
from flagquantum.algorithms.primitives.grid_kinetic import (
    source_grid_kinetic_cost,
    source_grid_kinetic_signature,
)
from flagquantum.algorithms.primitives.grid_potential_resource_model import (
    source_grid_pair_potential_cost,
    source_grid_pair_potential_signature,
    source_grid_potential_cost,
    source_grid_potential_signature,
)
from flagquantum.algorithms.primitives.grid_qvr import (
    source_grid_qvr_cost,
    source_grid_qvr_signature,
)
from examples.algorithms.grid_hamiltonian_resources import grid_tutorial_call_graph
from flagquantum.drawer.call_graph import (
    export_call_graph_dot,
    format_call_graph_markdown,
    render_call_graph_svg,
)
from flagquantum.drawer.signature_graph import export_signature_graph_dot


def show_signature(signature, model):
    display(SVG(data=render_call_graph_svg(
        export_signature_graph_dot(signature, model.name, pretty=True)
    ).decode()))


def show_calls(model):
    graph = grid_tutorial_call_graph(model)
    display(SVG(data=render_call_graph_svg(
        export_call_graph_dot(graph.root, graph.direct_calls, title=lambda node: node.name)
    ).decode()))
    display(Markdown(format_call_graph_markdown(
        graph.root, graph.direct_calls, graph.leaf_counts, title=lambda node: node.name
    )))

## Polynomial inverse square root

The squared distance and four coefficient words feed three real products and three adds. The coefficient bank is a shaped through port.

### Register signature

In [ ]:
signature = source_grid_inverse_sqrt_polynomial_signature(7, 8, 12)
model = source_grid_inverse_sqrt_polynomial_cost(7, 8, 12)
show_signature(signature, model)

### Immediate calls and leaf counts

In [ ]:
show_calls(model)

## Newton inverse square root

A square, scale, two real products and a signed add refine the cubic estimate. The three opaque through ports specify distinct widths.

### Register signature

In [ ]:
signature = source_grid_inverse_sqrt_newton_signature(7, 8, 12)
model = source_grid_inverse_sqrt_newton_cost(7, 8, 12)
show_signature(signature, model)

### Immediate calls and leaf counts

In [ ]:
show_calls(model)

## Quantum variable rotation

The 12-bit phase word stays live across 12 symbolic rotations. The original model leaves the rotation angle unspecified.

### Register signature

In [ ]:
signature = source_grid_qvr_signature(12)
model = source_grid_qvr_cost(12)
show_signature(signature, model)

### Immediate calls and leaf counts

In [ ]:
show_calls(model)

## Kinetic energy

On 12 electrons at 17 grid points per axis, each momentum row supplies three unsigned coordinates for the square, phase and release chain.

### Register signature

In [ ]:
signature = source_grid_kinetic_signature(12, 17)
model = source_grid_kinetic_cost(12, 17)
show_signature(signature, model)

### Immediate calls and leaf counts

In [ ]:
show_calls(model)

## Pair potential

Two three-axis registers supply the pair calculation. The locked 7-bit coordinate example uses a four-bank 15-bit coefficient table and a 24-bit estimate.

### Register signature

In [ ]:
signature = source_grid_pair_potential_signature(7)
model = source_grid_pair_potential_cost(7, 15, 24)
show_signature(signature, model)

### Immediate calls and leaf counts

In [ ]:
show_calls(model)

## Total potential

The 12-electron example traverses the 66 unordered pairs in upper-triangular order and preserves the shaped system register.

### Register signature

In [ ]:
signature = source_grid_potential_signature(12, 17)
model = source_grid_potential_cost(12, 17)
show_signature(signature, model)

### Immediate calls and leaf counts

In [ ]:
show_calls(model)

## Append a small kinetic term

Set a numerical phase coefficient and bind three one-bit coordinates, four sum
wires and one work wire. The clean dynamic execution uses the shared square and
phase emitters. Its measured instructions and the declared QEC estimates use
separate resource columns.

In [ ]:
from flagquantum.algorithms.primitives.grid_kinetic import GridKineticEnergy
from flagquantum.dynamic import DynamicCircuit
from flagquantum.qec.resources.estimation import estimate_ftqc_resources

op = GridKineticEnergy.make_on(
    grid_points=1,
    system=(((0,), (1,), (2,)),),
    result=(3, 4, 5, 6),
    workspace=(7,),
    radians_per_unit=0.317,
)
circuit = DynamicCircuit(8).append(op)
report = estimate_ftqc_resources(op, emitted_ir=circuit.to_ir())
assert report.emitted is not None
assert report.emitted.t_count == 8
assert report.emitted.unconditional["measure"] == 2
assert report.logical.gates.toffoli == 1
print("Emitted T-family instructions:", report.emitted.t_count)
print("Emitted measurements:", report.emitted.unconditional["measure"])
print("Modeled Toffoli equivalents:", report.logical.gates.toffoli)